# 010 - Extração e Transformação dos Dados

## Objetivo
Preparar as bases de dados para os relatórios estaduais. Este notebook não gera o documento Word final; ele transforma os arquivos de origem em CSVs intermediários, padronizados e rastreáveis.

## Quando executar
Execute este notebook sempre que houver atualização dos arquivos brutos atuais ou históricos.

## Entradas principais
- `dados_brutos/dado_atual`: arquivos Excel atuais das políticas.
- `dados_brutos/dado_historico/PRONAF`: série histórica do PRONAF.
- `dados_brutos/dado_historico/ATER`: série histórica de ATER.
- `config/ufs.csv`: tabela oficial de UFs usada para padronização.

## Saídas principais
- `dados_intermediarios/<AAAAMM>/politicas/`: CSVs limpos por política.
- `dados_intermediarios/<AAAAMM>/historico/`: CSVs históricos tratados.
- `dados_intermediarios/<AAAAMM>/consolidado/indicadores_brasil_uf.csv`: indicadores atuais por UF e Brasil.
- `dados_intermediarios/<AAAAMM>/consolidado/series_historicas_pronaf_ater_uf.csv`: séries históricas por UF.
- `dados_intermediarios/<AAAAMM>/logs/`: inventários e arquivos de rastreabilidade.

## Princípios desta etapa
- tratar cada política em seção própria;
- salvar resultados intermediários em CSV;
- calcular indicadores nacionais, estaduais e ranking das UFs;
- manter os nomes de campos consistentes para consumo pelos notebooks `020` e `030`.

## 1. Importar bibliotecas

As bibliotecas abaixo cobrem leitura de Excel, manipulacao tabular, caminhos de arquivos e tratamento pontual de nomes de colunas.

In [55]:
from datetime import datetime
from pathlib import Path
import os
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 2. Configurar parâmetros de execução

O modo é escolhido automaticamente. No VS Code/local, o notebook usa a raiz local do projeto. No Colab, ele monta o Google Drive e procura os dados nos atalhos/pastas configurados.

Os CSVs usam separador `;` e codificação `utf-8-sig`, formato mais amigável para abertura direta no Excel em ambiente brasileiro.

In [56]:
MODO_DADOS = "google_drive" if "COLAB_RELEASE_TAG" in os.environ else "local"

# Raiz local informada para execucao no VS Code/local.
LOCAL_PROJECT_ROOT = Path("C:/Users/marce/OneDrive - Ministério da Agricultura e Pecuária/LAB_DATA_MDA/20260424_relatorio_dados/relatorio_dados_damei")

# IDs/caminhos usados no Google Drive. O ID abaixo vem da pasta dado_historico mostrada no Drive.
GOOGLE_DRIVE_HISTORICO_FOLDER_ID = "1C0fp4QITO6EhzODQu_KQMZWxXyfAPI0n"
GOOGLE_DRIVE_EXCEL_FOLDER_ID = "1xGcxCFmkeRrETVpePgzzdvleaWKWr0IO"
GOOGLE_DRIVE_RAW_ATUAL_DIR = Path("/content/drive/MyDrive/DAMEI_Relatorio_Dados atalho/dados_brutos/dado_atual")
GOOGLE_DRIVE_OUTPUT_INTERMEDIARIOS_DIR = Path("/content/drive/MyDrive/MDA/dados_intermediarios")
RAW_ATUAL_DIR_OVERRIDE = os.environ.get("RAW_ATUAL_DIR") or None

RAW_ATUAL_SUBDIR = Path("dados_brutos") / "dado_atual"
RAW_ATUAL_PREPARADO_NOME = "dados_atuais"
HISTORICO_SUBDIR = Path("dados_brutos") / "dado_historico"
INTERMEDIARIOS_SUBDIR = Path("dados_intermediarios")
UF_TABLE_SUBPATH = Path("config") / "ufs.csv"

RUN_TIMESTAMP = datetime.now()
RUN_YYYYMM = RUN_TIMESTAMP.strftime("%Y%m")
RUN_DATETIME = RUN_TIMESTAMP.strftime("%Y%m%d%H%M%S")

CSV_SEP = ";"
CSV_ENCODING = "utf-8-sig"

print(f"Modo de dados: {MODO_DADOS}")
print(f"Data/hora da execucao: {RUN_TIMESTAMP:%d/%m/%Y %H:%M:%S}")
print(f"Override dados atuais: {RAW_ATUAL_DIR_OVERRIDE or 'nao informado'}")

Modo de dados: local
Data/hora da execucao: 14/05/2026 16:35:51
Override dados atuais: nao informado


## 3. Resolver caminhos de entrada e saída

Esta etapa localiza a raiz do código, a pasta de dados atuais, a pasta de históricos e as pastas de saída.

Em execução local, os históricos esperados ficam em `dados_brutos/dado_historico/PRONAF` e `dados_brutos/dado_historico/ATER`. Em Colab, o notebook procura primeiro os atalhos do Google Drive.

In [57]:
inicio = Path.cwd().resolve()
CODE_PROJECT_ROOT = inicio

for candidato in [inicio, *inicio.parents]:
    tem_notebooks = (candidato / "notebooks").exists()
    tem_requirements = (candidato / "requirements.txt").exists()
    tem_config = (candidato / UF_TABLE_SUBPATH).exists()
    if tem_notebooks and (tem_requirements or tem_config):
        CODE_PROJECT_ROOT = candidato
        break

if MODO_DADOS == "local":
    DATA_PROJECT_ROOT = LOCAL_PROJECT_ROOT if LOCAL_PROJECT_ROOT.exists() else CODE_PROJECT_ROOT

    candidatos_atual = []
    if RAW_ATUAL_DIR_OVERRIDE:
        candidatos_atual.append(Path(RAW_ATUAL_DIR_OVERRIDE).expanduser())
    candidatos_atual.extend([
        DATA_PROJECT_ROOT / RAW_ATUAL_SUBDIR / RAW_ATUAL_PREPARADO_NOME,
        CODE_PROJECT_ROOT / RAW_ATUAL_SUBDIR / RAW_ATUAL_PREPARADO_NOME,
        DATA_PROJECT_ROOT / RAW_ATUAL_SUBDIR,
        CODE_PROJECT_ROOT / RAW_ATUAL_SUBDIR,
        DATA_PROJECT_ROOT / "dado_atual",
        DATA_PROJECT_ROOT,
    ])
    RAW_ATUAL_DIR = None
    for candidato in candidatos_atual:
        if candidato.exists() and candidato.is_dir() and list(candidato.glob("*.xlsx")):
            RAW_ATUAL_DIR = candidato
            break
    if RAW_ATUAL_DIR is None:
        RAW_ATUAL_DIR = DATA_PROJECT_ROOT / RAW_ATUAL_SUBDIR

    HISTORICO_ROOT_DIR = DATA_PROJECT_ROOT / HISTORICO_SUBDIR
    OUTPUT_ROOT_DIR = CODE_PROJECT_ROOT / INTERMEDIARIOS_SUBDIR / RUN_YYYYMM

elif MODO_DADOS == "google_drive":
    from google.colab import drive

    drive.mount("/content/drive")
    atalhos = Path("/content/drive/MyDrive/.shortcut-targets-by-id")

    candidatos_atual = []
    if RAW_ATUAL_DIR_OVERRIDE:
        candidatos_atual.append(Path(RAW_ATUAL_DIR_OVERRIDE).expanduser())
    candidatos_atual.extend([
        atalhos / GOOGLE_DRIVE_EXCEL_FOLDER_ID,
        GOOGLE_DRIVE_RAW_ATUAL_DIR / RAW_ATUAL_PREPARADO_NOME,
        GOOGLE_DRIVE_RAW_ATUAL_DIR,
        Path("/content/drive/MyDrive/DAMEI_Relatorio_Dados/dados_brutos/dado_atual") / RAW_ATUAL_PREPARADO_NOME,
        Path("/content/drive/MyDrive/DAMEI_Relatorio_Dados/dados_brutos/dado_atual"),
        Path("/content/drive/MyDrive/MDA/DAMEI_Relatorio_Dados/dados_brutos/dado_atual") / RAW_ATUAL_PREPARADO_NOME,
        Path("/content/drive/MyDrive/MDA/DAMEI_Relatorio_Dados/dados_brutos/dado_atual"),
    ])
    RAW_ATUAL_DIR = None
    for candidato in candidatos_atual:
        if candidato.exists() and candidato.is_dir() and list(candidato.glob("*.xlsx")):
            RAW_ATUAL_DIR = candidato
            break
    if RAW_ATUAL_DIR is None:
        raise FileNotFoundError("Nao foi possivel localizar os arquivos atuais no Google Drive.")

    candidatos_historico = [
        atalhos / GOOGLE_DRIVE_HISTORICO_FOLDER_ID,
        RAW_ATUAL_DIR.parent / "dado_historico",
        Path("/content/drive/MyDrive/DAMEI_Relatorio_Dados/dados_brutos/dado_historico"),
        Path("/content/drive/MyDrive/MDA/DAMEI_Relatorio_Dados/dados_brutos/dado_historico"),
    ]
    HISTORICO_ROOT_DIR = None
    for candidato in candidatos_historico:
        if candidato.exists() and candidato.is_dir():
            HISTORICO_ROOT_DIR = candidato
            break
    if HISTORICO_ROOT_DIR is None:
        HISTORICO_ROOT_DIR = RAW_ATUAL_DIR.parent / "dado_historico"

    DATA_PROJECT_ROOT = RAW_ATUAL_DIR.parent.parent if RAW_ATUAL_DIR.parent.name == "dado_atual" else RAW_ATUAL_DIR.parent
    OUTPUT_ROOT_DIR = GOOGLE_DRIVE_OUTPUT_INTERMEDIARIOS_DIR / RUN_YYYYMM

else:
    raise ValueError('MODO_DADOS deve ser "local" ou "google_drive".')

HISTORICO_PRONAF_DIR = HISTORICO_ROOT_DIR / "PRONAF"
HISTORICO_ATER_DIR = HISTORICO_ROOT_DIR / "ATER"

POLITICAS_DIR = OUTPUT_ROOT_DIR / "politicas"
HISTORICO_DIR = OUTPUT_ROOT_DIR / "historico"
CONSOLIDADO_DIR = OUTPUT_ROOT_DIR / "consolidado"
LOGS_DIR = OUTPUT_ROOT_DIR / "logs"

for pasta in [POLITICAS_DIR, HISTORICO_DIR, CONSOLIDADO_DIR, LOGS_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

UF_TABLE_PATH = CODE_PROJECT_ROOT / UF_TABLE_SUBPATH

print(f"Raiz do codigo:       {CODE_PROJECT_ROOT}")
print(f"Raiz dos dados:       {DATA_PROJECT_ROOT}")
print(f"Dados atuais:         {RAW_ATUAL_DIR}")
print(f"Historico PRONAF:     {HISTORICO_PRONAF_DIR}")
print(f"Historico ATER:       {HISTORICO_ATER_DIR}")
print(f"Saida intermediaria:  {OUTPUT_ROOT_DIR}")

Raiz do codigo:       C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei
Raiz dos dados:       C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei
Dados atuais:         C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_atual\dados_atuais
Historico PRONAF:     C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_historico\PRONAF
Historico ATER:       C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_historico\ATER
Saida intermediaria:  C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_intermediarios\2026

## 4. Inventário dos arquivos encontrados

Esta etapa identifica os arquivos atuais por política pública e lista os arquivos históricos de PRONAF e ATER. O inventário é salvo em CSV para rastreabilidade.

In [58]:
if not RAW_ATUAL_DIR.exists():
    raise FileNotFoundError(f"Pasta de dados atuais nao encontrada: {RAW_ATUAL_DIR}")

excel_files_atuais = sorted(RAW_ATUAL_DIR.glob("*.xlsx"))
if not excel_files_atuais:
    raise FileNotFoundError(f"Nenhum arquivo .xlsx encontrado em {RAW_ATUAL_DIR}")

arquivos_politicas = {
    "caf_dap": None,
    "ater": None,
    "pronaf": None,
    "mais_alimentos": None,
    "garantia_safra": None,
    "pncf": None,
    "pnra": None,
}

for arquivo in excel_files_atuais:
    nome = arquivo.name.lower()
    if "caf" in nome:
        arquivos_politicas["caf_dap"] = arquivo
    elif "ater" in nome:
        arquivos_politicas["ater"] = arquivo
    elif "pronaf" in nome:
        arquivos_politicas["pronaf"] = arquivo
    elif "mais_alimentos" in nome or "mais-alimentos" in nome:
        arquivos_politicas["mais_alimentos"] = arquivo
    elif "garantia-safra" in nome or "garantia_safra" in nome:
        arquivos_politicas["garantia_safra"] = arquivo
    elif "pncf" in nome:
        arquivos_politicas["pncf"] = arquivo
    elif "pnra" in nome:
        arquivos_politicas["pnra"] = arquivo

pronaf_historico_files = sorted(HISTORICO_PRONAF_DIR.glob("*.xlsx")) if HISTORICO_PRONAF_DIR.exists() else []
ater_historico_files = sorted(HISTORICO_ATER_DIR.glob("*.xlsx")) if HISTORICO_ATER_DIR.exists() else []

linhas_inventario = []
for politica, arquivo in arquivos_politicas.items():
    linhas_inventario.append({
        "tipo": "atual",
        "politica": politica,
        "arquivo": arquivo.name if arquivo else "NAO ENCONTRADO",
        "caminho": str(arquivo) if arquivo else "",
        "encontrado": arquivo is not None,
    })

for arquivo in pronaf_historico_files:
    linhas_inventario.append({"tipo": "historico", "politica": "pronaf", "arquivo": arquivo.name, "caminho": str(arquivo), "encontrado": True})
for arquivo in ater_historico_files:
    linhas_inventario.append({"tipo": "historico", "politica": "ater", "arquivo": arquivo.name, "caminho": str(arquivo), "encontrado": True})

df_inventario = pd.DataFrame(linhas_inventario)
df_inventario.to_csv(LOGS_DIR / "inventario_arquivos.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(f"Arquivos atuais encontrados: {len(excel_files_atuais)}")
print(f"Historicos PRONAF encontrados: {len(pronaf_historico_files)}")
print(f"Historicos ATER encontrados: {len(ater_historico_files)}")
display(df_inventario)

Arquivos atuais encontrados: 7
Historicos PRONAF encontrados: 11
Historicos ATER encontrados: 9


,tipo,politica,arquivo,caminho,encontrado
0,atual,caf_dap,caf_dap_ativos_ate_2026_04_gerado_em_202605121...,C:\Users\marce\OneDrive - Ministério da Agricu...,True
1,atual,ater,ater_ate_2026_04_gerado_em_20260512153352.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True
2,atual,pronaf,pronaf_gaia_202605131604.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True
3,atual,mais_alimentos,mais_alimentos_gaia_202605131504.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True
4,atual,garantia_safra,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True
5,atual,pncf,pncf_2026_04_gerado_em_20260514095754.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True
6,atual,pnra,PNRA_2026_2026_04_15.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True
7,historico,pronaf,pronaf_gaia_historico_anual_2015.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True
8,historico,pronaf,pronaf_gaia_historico_anual_2016.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True
9,historico,pronaf,pronaf_gaia_historico_anual_2017.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,True


## 5. Inspecionar abas e colunas

A inspeção abaixo gera um CSV com abas e primeiras colunas de cada arquivo. Isso ajuda a auditar mudancas de layout nas planilhas de origem.

In [59]:
linhas_estrutura = []

for politica, arquivo in arquivos_politicas.items():
    if arquivo is None:
        linhas_estrutura.append({"tipo": "atual", "politica": politica, "arquivo": "NAO ENCONTRADO", "aba": "", "qtd_colunas": 0, "colunas": ""})
        continue
    xls = pd.ExcelFile(arquivo)
    for aba in xls.sheet_names:
        if politica == "caf_dap" and aba == "TOTALIZADORES":
            colunas = pd.read_excel(arquivo, sheet_name=aba, nrows=0, skiprows=6).columns.astype(str).tolist()
        else:
            colunas = pd.read_excel(arquivo, sheet_name=aba, nrows=0).columns.astype(str).tolist()
        linhas_estrutura.append({
            "tipo": "atual",
            "politica": politica,
            "arquivo": arquivo.name,
            "aba": aba,
            "qtd_colunas": len(colunas),
            "colunas": ", ".join(colunas[:20]),
        })

for politica, lista_arquivos in [("pronaf", pronaf_historico_files), ("ater", ater_historico_files)]:
    for arquivo in lista_arquivos:
        xls = pd.ExcelFile(arquivo)
        for aba in xls.sheet_names:
            colunas = pd.read_excel(arquivo, sheet_name=aba, nrows=0).columns.astype(str).tolist()
            linhas_estrutura.append({
                "tipo": "historico",
                "politica": politica,
                "arquivo": arquivo.name,
                "aba": aba,
                "qtd_colunas": len(colunas),
                "colunas": ", ".join(colunas[:20]),
            })

df_estrutura = pd.DataFrame(linhas_estrutura)
df_estrutura.to_csv(LOGS_DIR / "estrutura_abas_colunas.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
display(df_estrutura)

,tipo,politica,arquivo,aba,qtd_colunas,colunas
0,atual,caf_dap,caf_dap_ativos_ate_2026_04_gerado_em_202605121...,Orientações,1,Orientações
1,atual,caf_dap,caf_dap_ativos_ate_2026_04_gerado_em_202605121...,DADOS,9,"dt_referencia, dt_geracao, cod_ibge, nome_muni..."
2,atual,caf_dap,caf_dap_ativos_ate_2026_04_gerado_em_202605121...,TOTALIZADORES,8,"dt_referencia, dt_geracao, cod_ibge_estados, u..."
3,atual,ater,ater_ate_2026_04_gerado_em_20260512153352.xlsx,Orientações,1,Orientações
4,atual,ater,ater_ate_2026_04_gerado_em_20260512153352.xlsx,DADOS,7,"dt_referencia, dt_geracao, cod_ibge, nome_muni..."
...,...,...,...,...,...,...
76,historico,ater,ater_ate_2025_12_gerado_em_20260114105014.xlsx,DADOS,7,"dt_referencia, dt_geracao, cod_ibge, nome_muni..."
77,historico,ater,ater_ate_2025_12_gerado_em_20260114105014.xlsx,TOTALIZADORES,1,Planilha de Dados - Totalizadores
78,historico,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,Orientações,1,Orientações
79,historico,ater,ater_ate_2026_03_gerado_em_20260410151127.xlsx,DADOS,7,"dt_referencia, dt_geracao, cod_ibge, nome_muni..."


## 6. Carregar tabela oficial de UFs

A tabela `config/ufs.csv` padroniza siglas, codigos IBGE e nomes das UFs. Ela sera usada para filtrar registros invalidos e para montar a base consolidada.

In [60]:
if not UF_TABLE_PATH.exists():
    raise FileNotFoundError(f"Tabela de UFs nao encontrada: {UF_TABLE_PATH}")

df_ufs = pd.read_csv(UF_TABLE_PATH, dtype={"sigla": "string", "codigo_ibge": "string", "nome_uf": "string"})
df_ufs["sigla"] = df_ufs["sigla"].astype(str).str.strip().str.upper()
df_ufs["codigo_ibge"] = df_ufs["codigo_ibge"].astype(str).str.strip().str.zfill(2)
df_ufs["nome_uf"] = df_ufs["nome_uf"].astype(str).str.strip()

UF_VALIDAS = set(df_ufs["sigla"].tolist())
df_ufs.to_csv(LOGS_DIR / "ufs_referencia.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
display(df_ufs)

,sigla,codigo_ibge,nome_uf
0,AC,12,Acre
1,AL,27,Alagoas
2,AP,16,Amapá
3,AM,13,Amazonas
4,BA,29,Bahia
5,CE,23,Ceará
6,DF,53,Distrito Federal
7,ES,32,Espírito Santo
8,GO,52,Goiás
9,MA,21,Maranhão


## 7. CAF/DAP - Limpeza e transformacao

Entradas: abas `DADOS` e `TOTALIZADORES` da base CAF/DAP atual.

Saidas:
- `caf_municipio_tratado.csv`
- `caf_uf_tratado.csv`

In [61]:
arquivo_caf = arquivos_politicas["caf_dap"]
if arquivo_caf is None:
    raise FileNotFoundError("Arquivo CAF/DAP nao encontrado.")

df_caf_municipio = pd.read_excel(arquivo_caf, sheet_name="DADOS", engine="openpyxl", na_values=["NA", "na", ""])
df_caf_municipio.columns = df_caf_municipio.columns.astype(str).str.strip()

caf_municipio_colunas_obrigatorias = [
    "dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf",
    "CAFs PF ATIVO", "CAFs PJ ATIVO", "QUANTIDADE DE MULHERES EM CAF ATIVO", "QUANTIDADE DE HOMENS EM CAF ATIVO",
]
faltantes = [col for col in caf_municipio_colunas_obrigatorias if col not in df_caf_municipio.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em CAF/DAP DADOS: {faltantes}")

df_caf_municipio = df_caf_municipio[caf_municipio_colunas_obrigatorias].copy()
# A coluna original "CAFs PF ATIVO" representa a UFPA e passa a ser padronizada como "ufpa"
# nos arquivos tratados, preservando a origem no arquivo bruto.
df_caf_municipio = df_caf_municipio.rename(columns={
    "CAFs PF ATIVO": "ufpa",
    "CAFs PJ ATIVO": "cafs_pj_ativo",
    "QUANTIDADE DE MULHERES EM CAF ATIVO": "mulheres_em_caf_ativo",
    "QUANTIDADE DE HOMENS EM CAF ATIVO": "homens_em_caf_ativo",
})

for col in ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf"]:
    df_caf_municipio[col] = df_caf_municipio[col].astype(str).str.strip()
for col in ["ufpa", "cafs_pj_ativo", "mulheres_em_caf_ativo", "homens_em_caf_ativo"]:
    df_caf_municipio[col] = pd.to_numeric(df_caf_municipio[col], errors="coerce").fillna(0)
df_caf_municipio["uf"] = df_caf_municipio["uf"].str.upper()
df_caf_municipio = df_caf_municipio[df_caf_municipio["uf"].isin(UF_VALIDAS)].copy()
df_caf_municipio["arquivo_origem"] = arquivo_caf.name

df_caf_uf = pd.read_excel(arquivo_caf, sheet_name="TOTALIZADORES", engine="openpyxl", skiprows=6, nrows=27, na_values=["NA", "na", ""])
df_caf_uf.columns = df_caf_uf.columns.astype(str).str.strip()
# Nos totalizadores, a mesma variavel vem como "Soma de CAFs PF ATIVO" e recebe o mesmo padrao.
df_caf_uf = df_caf_uf.rename(columns={
    "cod_ibge_estados": "cod_ibge_uf",
    "Soma de CAFs PF ATIVO": "ufpa",
    "Soma de CAFs PJ ATIVO": "cafs_pj_ativo",
    "Soma de QUANTIDADE DE MULHERES EM CAF ATIVO": "mulheres_em_caf_ativo",
    "Soma de QUANTIDADE DE HOMENS EM CAF ATIVO": "homens_em_caf_ativo",
})

caf_uf_colunas = ["dt_referencia", "dt_geracao", "cod_ibge_uf", "uf", "ufpa", "cafs_pj_ativo", "mulheres_em_caf_ativo", "homens_em_caf_ativo"]
faltantes = [col for col in caf_uf_colunas if col not in df_caf_uf.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em CAF/DAP TOTALIZADORES: {faltantes}")
df_caf_uf = df_caf_uf[caf_uf_colunas].copy()
for col in ["dt_referencia", "dt_geracao", "cod_ibge_uf", "uf"]:
    df_caf_uf[col] = df_caf_uf[col].astype(str).str.strip()
for col in ["ufpa", "cafs_pj_ativo", "mulheres_em_caf_ativo", "homens_em_caf_ativo"]:
    df_caf_uf[col] = pd.to_numeric(df_caf_uf[col], errors="coerce").fillna(0)
df_caf_uf["uf"] = df_caf_uf["uf"].str.upper()
df_caf_uf = df_caf_uf[df_caf_uf["uf"].isin(UF_VALIDAS)].copy()
df_caf_uf["arquivo_origem"] = arquivo_caf.name

df_caf_municipio.to_csv(POLITICAS_DIR / "caf_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_caf_uf.to_csv(POLITICAS_DIR / "caf_uf_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_caf_municipio.shape, df_caf_uf.shape)
display(df_caf_uf.head())

(5526, 10) (27, 9)


,dt_referencia,dt_geracao,cod_ibge_uf,uf,ufpa,cafs_pj_ativo,mulheres_em_caf_ativo,homens_em_caf_ativo,arquivo_origem
0,2026_04,2026_05_12,11,RO,59129,62,48861,57599,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
1,2026_04,2026_05_12,12,AC,33690,95,28756,33234,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
2,2026_04,2026_05_12,13,AM,66861,301,61082,65864,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
3,2026_04,2026_05_12,14,RR,14161,62,10729,11865,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
4,2026_04,2026_05_12,15,PA,220160,370,173748,183871,caf_dap_ativos_ate_2026_04_gerado_em_202605121...


In [62]:
# # teste de leitura do arquivo consolidado do caf/dap por município
# df_caf_municipio_teste = pd.read_csv(POLITICAS_DIR / "caf_municipio_tratado.csv", sep=CSV_SEP, encoding=CSV_ENCODING)
# print(df_caf_municipio_teste.shape)

# df_caf_municipio_teste

In [63]:
# # teste de leitura do arquivo consolidado do caf/dap por UF
# df_caf_uf_teste = pd.read_csv(POLITICAS_DIR / "caf_uf_tratado.csv", sep=CSV_SEP, encoding=CSV_ENCODING)
# print(df_caf_uf_teste.shape)

# df_caf_uf_teste

## 8. PRONAF atual - Limpeza e transformacao

Entradas: abas `Dados` e `Totalizadores` da base PRONAF atual.

Saidas:
- `pronaf_atual_municipio_tratado.csv`
- `pronaf_atual_uf_tratado.csv`

In [64]:
arquivo_pronaf = arquivos_politicas["pronaf"]
if arquivo_pronaf is None:
    raise FileNotFoundError("Arquivo PRONAF atual nao encontrado.")

df_pronaf_atual_municipio = pd.read_excel(arquivo_pronaf, sheet_name="Dados", engine="openpyxl", na_values=["NA", "na", ""])
df_pronaf_atual_municipio.columns = df_pronaf_atual_municipio.columns.astype(str).str.strip()

df_pronaf_atual_uf = pd.read_excel(arquivo_pronaf, sheet_name="Totalizadores", engine="openpyxl", na_values=["NA", "na", ""])
df_pronaf_atual_uf.columns = df_pronaf_atual_uf.columns.astype(str).str.strip()

pronaf_texto_municipio = ["dt_referencia", "dt_geracao", "ANO", "nome_municipio", "uf", "cod_ibge_municipio", "cod_ibge_uf"]
pronaf_numericas = [
    "qtd_contratos_1_Feminino", "qtd_contratos_1_Masculino", "qtd_contratos_1_Sem_Identificacao",
    "valor_total_contratos_1_Feminino", "valor_total_contratos_1_Masculino", "valor_total_contratos_1_Sem_Identificacao",
    "qtd_operacoes_1_Feminino", "qtd_operacoes_1_Masculino", "qtd_operacoes_1_Sem_Identificacao",
    "ticket_medio_1_Feminino", "ticket_medio_1_Masculino", "ticket_medio_1_Sem_Identificacao",
]

faltantes = [col for col in pronaf_texto_municipio + pronaf_numericas if col not in df_pronaf_atual_municipio.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em PRONAF Dados: {faltantes}")

for col in pronaf_texto_municipio:
    df_pronaf_atual_municipio[col] = df_pronaf_atual_municipio[col].astype(str).str.strip()
for col in pronaf_numericas:
    df_pronaf_atual_municipio[col] = pd.to_numeric(df_pronaf_atual_municipio[col], errors="coerce").fillna(0)
df_pronaf_atual_municipio["uf"] = df_pronaf_atual_municipio["uf"].str.upper()
df_pronaf_atual_municipio = df_pronaf_atual_municipio[df_pronaf_atual_municipio["uf"].isin(UF_VALIDAS)].copy()
df_pronaf_atual_municipio["ano"] = pd.to_numeric(df_pronaf_atual_municipio["ANO"], errors="coerce").astype("Int64")
df_pronaf_atual_municipio["qtd_contratos_total"] = df_pronaf_atual_municipio[["qtd_contratos_1_Feminino", "qtd_contratos_1_Masculino", "qtd_contratos_1_Sem_Identificacao"]].sum(axis=1)
df_pronaf_atual_municipio["valor_total_contratos"] = df_pronaf_atual_municipio[["valor_total_contratos_1_Feminino", "valor_total_contratos_1_Masculino", "valor_total_contratos_1_Sem_Identificacao"]].sum(axis=1)
df_pronaf_atual_municipio["qtd_operacoes_total"] = df_pronaf_atual_municipio[["qtd_operacoes_1_Feminino", "qtd_operacoes_1_Masculino", "qtd_operacoes_1_Sem_Identificacao"]].sum(axis=1)
df_pronaf_atual_municipio["arquivo_origem"] = arquivo_pronaf.name

pronaf_texto_uf = ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf", "ANO"]
faltantes = [col for col in pronaf_texto_uf + pronaf_numericas if col not in df_pronaf_atual_uf.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em PRONAF Totalizadores: {faltantes}")
for col in pronaf_texto_uf:
    df_pronaf_atual_uf[col] = df_pronaf_atual_uf[col].astype(str).str.strip()
for col in pronaf_numericas:
    df_pronaf_atual_uf[col] = pd.to_numeric(df_pronaf_atual_uf[col], errors="coerce").fillna(0)
df_pronaf_atual_uf["uf"] = df_pronaf_atual_uf["uf"].str.upper()
df_pronaf_atual_uf = df_pronaf_atual_uf[df_pronaf_atual_uf["uf"].isin(UF_VALIDAS)].copy()
df_pronaf_atual_uf["ano"] = pd.to_numeric(df_pronaf_atual_uf["ANO"], errors="coerce").astype("Int64")
df_pronaf_atual_uf["qtd_contratos_total"] = df_pronaf_atual_uf[["qtd_contratos_1_Feminino", "qtd_contratos_1_Masculino", "qtd_contratos_1_Sem_Identificacao"]].sum(axis=1)
df_pronaf_atual_uf["valor_total_contratos"] = df_pronaf_atual_uf[["valor_total_contratos_1_Feminino", "valor_total_contratos_1_Masculino", "valor_total_contratos_1_Sem_Identificacao"]].sum(axis=1)
df_pronaf_atual_uf["qtd_operacoes_total"] = df_pronaf_atual_uf[["qtd_operacoes_1_Feminino", "qtd_operacoes_1_Masculino", "qtd_operacoes_1_Sem_Identificacao"]].sum(axis=1)
df_pronaf_atual_uf["arquivo_origem"] = arquivo_pronaf.name

df_pronaf_atual_municipio.to_csv(POLITICAS_DIR / "pronaf_atual_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_pronaf_atual_uf.to_csv(POLITICAS_DIR / "pronaf_atual_uf_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_pronaf_atual_municipio.shape, df_pronaf_atual_uf.shape)
display(df_pronaf_atual_uf.head())

(4884, 24) (27, 22)


,dt_referencia,dt_geracao,uf,cod_ibge_uf,ANO,qtd_contratos_1_Feminino,qtd_contratos_1_Masculino,qtd_contratos_1_Sem_Identificacao,valor_total_contratos_1_Feminino,valor_total_contratos_1_Masculino,valor_total_contratos_1_Sem_Identificacao,qtd_operacoes_1_Feminino,qtd_operacoes_1_Masculino,qtd_operacoes_1_Sem_Identificacao,ticket_medio_1_Feminino,ticket_medio_1_Masculino,ticket_medio_1_Sem_Identificacao,ano,qtd_contratos_total,valor_total_contratos,qtd_operacoes_total,arquivo_origem
0,2026_04,2026_05_13,AC,12.0,2026.0,648,963,3512,3.291714e+07,7.050305e+07,8.485853e+07,648.0,963.0,3512.0,50798.051636,73211.886127,24162.450592,2026,5123,1.882787e+08,5123.0,pronaf_gaia_202605131604.xlsx
1,2026_04,2026_05_13,AL,27.0,2026.0,7278,8129,5147,8.771298e+07,1.224188e+08,1.054877e+08,7278.0,8129.0,5147.0,12051.797005,15059.519192,20494.995739,2026,20554,3.156196e+08,20554.0,pronaf_gaia_202605131604.xlsx
2,2026_04,2026_05_13,AM,13.0,2026.0,455,664,2442,6.869497e+06,1.297131e+07,3.522152e+07,455.0,664.0,2442.0,15097.795868,19535.098946,14423.227801,2026,3561,5.506233e+07,3561.0,pronaf_gaia_202605131604.xlsx
3,2026_04,2026_05_13,AP,16.0,2026.0,327,379,1923,7.449358e+06,8.008255e+06,2.620365e+07,327.0,379.0,1923.0,22780.910306,21129.961266,13626.444613,2026,2629,4.166127e+07,2629.0,pronaf_gaia_202605131604.xlsx
4,2026_04,2026_05_13,BA,29.0,2026.0,42223,44212,27685,4.484689e+08,7.027358e+08,4.592457e+08,42223.0,44212.0,27685.0,10621.435667,15894.685520,16588.251408,2026,114120,1.610450e+09,114120.0,pronaf_gaia_202605131604.xlsx


## 9. PRONAF historico - Limpeza e transformacao

Os historicos podem estar no local ou no Google Drive. O notebook le todos os `.xlsx` encontrados na pasta `PRONAF`, padroniza as colunas anuais e salva a serie consolidada.

Saidas:
- `pronaf_historico_municipio_tratado.csv`
- `pronaf_historico_anual_uf.csv`

In [65]:
lista_pronaf_historico = []

for arquivo in pronaf_historico_files:
    xls = pd.ExcelFile(arquivo)
    aba = "Dados" if "Dados" in xls.sheet_names else ("DADOS" if "DADOS" in xls.sheet_names else xls.sheet_names[0])
    df_temp = pd.read_excel(arquivo, sheet_name=aba, engine="openpyxl", na_values=["NA", "na", ""])
    df_temp.columns = df_temp.columns.astype(str).str.strip()
    df_temp = df_temp.rename(columns={
        "qtd_contratos_anual_Feminino": "qtd_contratos_1_Feminino",
        "qtd_contratos_anual_Masculino": "qtd_contratos_1_Masculino",
        "qtd_contratos_anual_Sem_Identificacao": "qtd_contratos_1_Sem_Identificacao",
        "valor_total_contratos_anual_Feminino": "valor_total_contratos_1_Feminino",
        "valor_total_contratos_anual_Masculino": "valor_total_contratos_1_Masculino",
        "valor_total_contratos_anual_Sem_Identificacao": "valor_total_contratos_1_Sem_Identificacao",
        "qtd_operacoes_anual_Feminino": "qtd_operacoes_1_Feminino",
        "qtd_operacoes_anual_Masculino": "qtd_operacoes_1_Masculino",
        "qtd_operacoes_anual_Sem_Identificacao": "qtd_operacoes_1_Sem_Identificacao",
    })
    if "ANO" not in df_temp.columns:
        achou_ano = re.search(r"(20\d{2})", arquivo.name)
        df_temp["ANO"] = int(achou_ano.group(1)) if achou_ano else pd.NA
    for col in ["dt_referencia", "dt_geracao", "ANO", "nome_municipio", "uf", "cod_ibge_municipio", "cod_ibge_uf"]:
        if col not in df_temp.columns:
            df_temp[col] = ""
        df_temp[col] = df_temp[col].astype(str).str.strip()
    for col in pronaf_numericas:
        if col not in df_temp.columns:
            df_temp[col] = 0
        df_temp[col] = pd.to_numeric(df_temp[col], errors="coerce").fillna(0)
    df_temp["uf"] = df_temp["uf"].str.upper()
    df_temp = df_temp[df_temp["uf"].isin(UF_VALIDAS)].copy()
    df_temp["ano"] = pd.to_numeric(df_temp["ANO"], errors="coerce").astype("Int64")
    df_temp["qtd_contratos_total"] = df_temp[["qtd_contratos_1_Feminino", "qtd_contratos_1_Masculino", "qtd_contratos_1_Sem_Identificacao"]].sum(axis=1)
    df_temp["valor_total_contratos"] = df_temp[["valor_total_contratos_1_Feminino", "valor_total_contratos_1_Masculino", "valor_total_contratos_1_Sem_Identificacao"]].sum(axis=1)
    df_temp["qtd_operacoes_total"] = df_temp[["qtd_operacoes_1_Feminino", "qtd_operacoes_1_Masculino", "qtd_operacoes_1_Sem_Identificacao"]].sum(axis=1)
    df_temp["arquivo_origem"] = arquivo.name
    lista_pronaf_historico.append(df_temp)

if lista_pronaf_historico:
    df_pronaf_historico_municipio = pd.concat(lista_pronaf_historico, ignore_index=True)
else:
    df_pronaf_historico_municipio = pd.DataFrame(columns=["dt_referencia", "dt_geracao", "ano", "nome_municipio", "uf", "cod_ibge_municipio", "cod_ibge_uf", "qtd_contratos_total", "valor_total_contratos", "qtd_operacoes_total", "arquivo_origem"])

df_pronaf_historico_anual_uf = (
    df_pronaf_historico_municipio
    .groupby(["ano", "uf"], as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        qtd_contratos_total=("qtd_contratos_total", "sum"),
        valor_total_contratos=("valor_total_contratos", "sum"),
        qtd_operacoes_total=("qtd_operacoes_total", "sum"),
        arquivo_origem=("arquivo_origem", "first"),
    )
    if not df_pronaf_historico_municipio.empty else pd.DataFrame(columns=["ano", "uf", "dt_referencia", "dt_geracao", "qtd_contratos_total", "valor_total_contratos", "qtd_operacoes_total", "arquivo_origem"])
)

df_pronaf_historico_municipio.to_csv(HISTORICO_DIR / "pronaf_historico_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_pronaf_historico_anual_uf.to_csv(HISTORICO_DIR / "pronaf_historico_anual_uf.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_pronaf_historico_municipio.shape, df_pronaf_historico_anual_uf.shape)
display(df_pronaf_historico_anual_uf.head())

(59072, 27) (297, 8)


,ano,uf,dt_referencia,dt_geracao,qtd_contratos_total,valor_total_contratos,qtd_operacoes_total,arquivo_origem
0,2015,AC,2015_12,2026_04_29,5501,1.110044e+08,5501,pronaf_gaia_historico_anual_2015.xlsx
1,2015,AL,2015_12,2026_04_29,50340,1.977084e+08,50340,pronaf_gaia_historico_anual_2015.xlsx
2,2015,AM,2015_12,2026_04_29,3382,5.350340e+07,3382,pronaf_gaia_historico_anual_2015.xlsx
3,2015,AP,2015_12,2026_04_29,804,1.102716e+07,804,pronaf_gaia_historico_anual_2015.xlsx
4,2015,BA,2015_12,2026_04_29,229983,8.123093e+08,229983,pronaf_gaia_historico_anual_2015.xlsx


In [66]:
# # teste de leitura do arquivo consolidado do pronaf historico por uf
# df_pronaf_historico_anual_uf_teste = pd.read_csv(HISTORICO_DIR / "pronaf_historico_anual_uf.csv", sep=CSV_SEP, encoding=CSV_ENCODING)
# print(df_pronaf_historico_anual_uf_teste.shape)

# df_pronaf_historico_anual_uf_teste

In [67]:
# # tabela com a soma por das uf para representar o Brasil como um todo, para o grafico de evolução anual do pronaf
# df_pronaf_historico_anual_brasil = (
#     df_pronaf_historico_anual_uf
#     .groupby("ano", as_index=False)
#     .agg(
#         dt_referencia=("dt_referencia", "first"),
#         dt_geracao=("dt_geracao", "first"),
#         qtd_contratos_total=("qtd_contratos_total", "sum"),
#         valor_total_contratos=("valor_total_contratos", "sum"),
#         qtd_operacoes_total=("qtd_operacoes_total", "sum"),
#         arquivo_origem=("arquivo_origem", "first"),
#     )
#     if not df_pronaf_historico_anual_uf.empty else pd.DataFrame(columns=["ano", "dt_referencia", "dt_geracao", "qtd_contratos_total", "valor_total_contratos", "qtd_operacoes_total", "arquivo_origem"])
# )
# print(df_pronaf_historico_anual_brasil)

## 10. Mais Alimentos - Limpeza e transformacao

Entradas: abas `Dados` e `Totalizadores` da base Mais Alimentos.

Saidas:
- `mais_alimentos_municipio_tratado.csv`
- `mais_alimentos_uf_tratado.csv`

In [68]:
arquivo_mais = arquivos_politicas["mais_alimentos"]
if arquivo_mais is None:
    raise FileNotFoundError("Arquivo Mais Alimentos nao encontrado.")

df_mais_municipio = pd.read_excel(arquivo_mais, sheet_name="Dados", engine="openpyxl", na_values=["NA", "na", ""])
df_mais_municipio.columns = df_mais_municipio.columns.astype(str).str.strip()
df_mais_municipio = df_mais_municipio.rename(columns={"CD_ESTADO": "uf"})

mais_municipio_colunas = ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf", "nome_municipio", "cod_ibge_municipio", "qtd_contratos", "valor_total_contratos"]
faltantes = [col for col in mais_municipio_colunas if col not in df_mais_municipio.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em Mais Alimentos Dados: {faltantes}")
df_mais_municipio = df_mais_municipio[mais_municipio_colunas].copy()
for col in ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf", "nome_municipio", "cod_ibge_municipio"]:
    df_mais_municipio[col] = df_mais_municipio[col].astype(str).str.strip()
for col in ["qtd_contratos", "valor_total_contratos"]:
    df_mais_municipio[col] = pd.to_numeric(df_mais_municipio[col], errors="coerce").fillna(0)
df_mais_municipio["uf"] = df_mais_municipio["uf"].str.upper()
df_mais_municipio = df_mais_municipio[df_mais_municipio["uf"].isin(UF_VALIDAS)].copy()
df_mais_municipio["arquivo_origem"] = arquivo_mais.name

df_mais_uf = pd.read_excel(arquivo_mais, sheet_name="Totalizadores", engine="openpyxl", na_values=["NA", "na", ""])
df_mais_uf.columns = df_mais_uf.columns.astype(str).str.strip()
df_mais_uf = df_mais_uf.rename(columns={"CD_ESTADO": "uf"})
mais_uf_colunas = ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf", "qtd_contratos", "valor_total_contratos"]
faltantes = [col for col in mais_uf_colunas if col not in df_mais_uf.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em Mais Alimentos Totalizadores: {faltantes}")
df_mais_uf = df_mais_uf[mais_uf_colunas].copy()
for col in ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf"]:
    df_mais_uf[col] = df_mais_uf[col].astype(str).str.strip()
for col in ["qtd_contratos", "valor_total_contratos"]:
    df_mais_uf[col] = pd.to_numeric(df_mais_uf[col], errors="coerce").fillna(0)
df_mais_uf["uf"] = df_mais_uf["uf"].str.upper()
df_mais_uf = df_mais_uf[df_mais_uf["uf"].isin(UF_VALIDAS)].copy()
df_mais_uf["arquivo_origem"] = arquivo_mais.name

df_mais_municipio.to_csv(POLITICAS_DIR / "mais_alimentos_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_mais_uf.to_csv(POLITICAS_DIR / "mais_alimentos_uf_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_mais_municipio.shape, df_mais_uf.shape)
display(df_mais_uf.head())

(4082, 9) (27, 7)


,dt_referencia,dt_geracao,uf,cod_ibge_uf,qtd_contratos,valor_total_contratos,arquivo_origem
0,2026_04,2026_05_13,AC,12,1552,2.220815e+07,mais_alimentos_gaia_202605131504.xlsx
1,2026_04,2026_05_13,AL,27,2919,1.765436e+07,mais_alimentos_gaia_202605131504.xlsx
2,2026_04,2026_05_13,AM,13,2042,2.404128e+07,mais_alimentos_gaia_202605131504.xlsx
3,2026_04,2026_05_13,AP,16,1554,2.020517e+07,mais_alimentos_gaia_202605131504.xlsx
4,2026_04,2026_05_13,BA,29,17847,1.891413e+08,mais_alimentos_gaia_202605131504.xlsx


## 11. ATER atual - Limpeza e transformacao

Entrada: aba `DADOS` da base ATER atual.

Saidas:
- `ater_atual_municipio_tratado.csv`
- `ater_atual_uf_tratado.csv`

In [69]:
arquivo_ater = arquivos_politicas["ater"]
if arquivo_ater is None:
    raise FileNotFoundError("Arquivo ATER atual nao encontrado.")

df_ater_atual_municipio = pd.read_excel(arquivo_ater, sheet_name="DADOS", engine="openpyxl", na_values=["NA", "na", ""])
df_ater_atual_municipio.columns = df_ater_atual_municipio.columns.astype(str).str.strip()
ater_colunas = ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf", "familias_com_ater_iniciada_no_ano", "familias_com_ater_recebida_no_ano"]
faltantes = [col for col in ater_colunas if col not in df_ater_atual_municipio.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em ATER atual: {faltantes}")
df_ater_atual_municipio = df_ater_atual_municipio[ater_colunas].copy()
for col in ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf"]:
    df_ater_atual_municipio[col] = df_ater_atual_municipio[col].astype(str).str.strip()
for col in ["familias_com_ater_iniciada_no_ano", "familias_com_ater_recebida_no_ano"]:
    df_ater_atual_municipio[col] = pd.to_numeric(df_ater_atual_municipio[col], errors="coerce").fillna(0)
df_ater_atual_municipio["uf"] = df_ater_atual_municipio["uf"].str.upper()
df_ater_atual_municipio = df_ater_atual_municipio[df_ater_atual_municipio["uf"].isin(UF_VALIDAS)].copy()
df_ater_atual_municipio["arquivo_origem"] = arquivo_ater.name

df_ater_atual_uf = (
    df_ater_atual_municipio
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        familias_com_ater_iniciada_no_ano=("familias_com_ater_iniciada_no_ano", "sum"),
        familias_com_ater_recebida_no_ano=("familias_com_ater_recebida_no_ano", "sum"),
        arquivo_origem=("arquivo_origem", "first"),
    )
)

df_ater_atual_municipio.to_csv(POLITICAS_DIR / "ater_atual_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_ater_atual_uf.to_csv(POLITICAS_DIR / "ater_atual_uf_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_ater_atual_municipio.shape, df_ater_atual_uf.shape)
display(df_ater_atual_uf.head())

(357, 8) (25, 6)


,uf,dt_referencia,dt_geracao,familias_com_ater_iniciada_no_ano,familias_com_ater_recebida_no_ano,arquivo_origem
0,AL,2026_04,20260512,143,180,ater_ate_2026_04_gerado_em_20260512153352.xlsx
1,AM,2026_04,20260512,0,178,ater_ate_2026_04_gerado_em_20260512153352.xlsx
2,AP,2026_04,20260512,411,967,ater_ate_2026_04_gerado_em_20260512153352.xlsx
3,BA,2026_04,20260512,147,790,ater_ate_2026_04_gerado_em_20260512153352.xlsx
4,CE,2026_04,20260512,1,506,ater_ate_2026_04_gerado_em_20260512153352.xlsx


## 12. ATER historico - Limpeza e transformacao

Os historicos podem estar no local ou no Google Drive. A etapa preserva os dois indicadores principais: ATER iniciada e ATER recebida.

Saidas:
- `ater_historico_municipio_tratado.csv`
- `ater_historico_anual_uf.csv`

In [70]:
lista_ater_historico = []

for arquivo in ater_historico_files:
    xls = pd.ExcelFile(arquivo)
    aba = "DADOS" if "DADOS" in xls.sheet_names else ("Dados" if "Dados" in xls.sheet_names else xls.sheet_names[0])
    df_temp = pd.read_excel(arquivo, sheet_name=aba, engine="openpyxl", na_values=["NA", "na", ""])
    df_temp.columns = df_temp.columns.astype(str).str.strip()
    df_temp = df_temp.rename(columns={
        "ano": "ANO",
        "Soma de familias_com_ater_iniciada_no_ano": "familias_com_ater_iniciada_no_ano",
        "Soma de familias_com_ater_recebida_no_ano": "familias_com_ater_recebida_no_ano",
        "familias_com_ater_iniciada_no_ano": "familias_com_ater_iniciada_no_ano",
        "familias_com_ater_recebida_no_ano": "familias_com_ater_recebida_no_ano",
    })
    if "ANO" not in df_temp.columns:
        achou_ano = re.search(r"(20\d{2})", arquivo.name)
        df_temp["ANO"] = int(achou_ano.group(1)) if achou_ano else pd.NA
    for col in ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf"]:
        if col not in df_temp.columns:
            df_temp[col] = ""
        df_temp[col] = df_temp[col].astype(str).str.strip()
    for col in ["familias_com_ater_iniciada_no_ano", "familias_com_ater_recebida_no_ano"]:
        if col not in df_temp.columns:
            df_temp[col] = 0
        df_temp[col] = pd.to_numeric(df_temp[col], errors="coerce").fillna(0)
    df_temp["uf"] = df_temp["uf"].str.upper()
    df_temp = df_temp[df_temp["uf"].isin(UF_VALIDAS)].copy()
    df_temp["ano"] = pd.to_numeric(df_temp["ANO"], errors="coerce").astype("Int64")
    df_temp["arquivo_origem"] = arquivo.name
    lista_ater_historico.append(df_temp)

if lista_ater_historico:
    df_ater_historico_municipio = pd.concat(lista_ater_historico, ignore_index=True)
else:
    df_ater_historico_municipio = pd.DataFrame(columns=["dt_referencia", "dt_geracao", "ano", "cod_ibge", "nome_municipio", "uf", "familias_com_ater_iniciada_no_ano", "familias_com_ater_recebida_no_ano", "arquivo_origem"])

df_ater_historico_anual_uf = (
    df_ater_historico_municipio
    .groupby(["ano", "uf"], as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        familias_com_ater_iniciada_no_ano=("familias_com_ater_iniciada_no_ano", "sum"),
        familias_com_ater_recebida_no_ano=("familias_com_ater_recebida_no_ano", "sum"),
        arquivo_origem=("arquivo_origem", "first"),
    )
    if not df_ater_historico_municipio.empty else pd.DataFrame(columns=["ano", "uf", "dt_referencia", "dt_geracao", "familias_com_ater_iniciada_no_ano", "familias_com_ater_recebida_no_ano", "arquivo_origem"])
)

df_ater_historico_municipio.to_csv(HISTORICO_DIR / "ater_historico_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_ater_historico_anual_uf.to_csv(HISTORICO_DIR / "ater_historico_anual_uf.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_ater_historico_municipio.shape, df_ater_historico_anual_uf.shape)
display(df_ater_historico_anual_uf.head())

(7174, 12) (208, 7)


,ano,uf,dt_referencia,dt_geracao,familias_com_ater_iniciada_no_ano,familias_com_ater_recebida_no_ano,arquivo_origem
0,2018,AL,2025-11-30,2025-12-12,2084,2095,ater_ano_2018_ref_2025-11-30_gerado_em_2025121...
1,2018,BA,2025-11-30,2025-12-12,4468,4468,ater_ano_2018_ref_2025-11-30_gerado_em_2025121...
2,2018,CE,2025-11-30,2025-12-12,6918,6920,ater_ano_2018_ref_2025-11-30_gerado_em_2025121...
3,2018,DF,2025-11-30,2025-12-12,439,500,ater_ano_2018_ref_2025-11-30_gerado_em_2025121...
4,2018,ES,2025-11-30,2025-12-12,1326,1327,ater_ano_2018_ref_2025-11-30_gerado_em_2025121...


## 13. Garantia-Safra - Limpeza e transformacao

Entrada: aba `DADOS` da base Garantia-Safra.

Saidas:
- `garantia_safra_municipio_tratado.csv`
- `garantia_safra_uf_tratado.csv`

In [71]:
arquivo_gs = arquivos_politicas["garantia_safra"]
if arquivo_gs is None:
    raise FileNotFoundError("Arquivo Garantia-Safra nao encontrado.")

df_gs_municipio = pd.read_excel(arquivo_gs, sheet_name="DADOS", engine="openpyxl", na_values=["NA", "na", ""])
df_gs_municipio.columns = df_gs_municipio.columns.astype(str).str.strip()
df_gs_municipio = df_gs_municipio.rename(columns={"Agricultores_aderidos": "agricultores_aderidos", "Agricultores_com_pagamento liberado": "agricultores_com_pagamento_liberado"})

gs_colunas = ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf", "cod_ibge", "nome_municipio", "agricultores_aderidos", "agricultores_com_pagamento_liberado", "pagamento_liberado"]
faltantes = [col for col in gs_colunas if col not in df_gs_municipio.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em Garantia-Safra: {faltantes}")
df_gs_municipio = df_gs_municipio[gs_colunas].copy()
for col in ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf", "cod_ibge", "nome_municipio", "pagamento_liberado"]:
    df_gs_municipio[col] = df_gs_municipio[col].astype(str).str.strip()
for col in ["agricultores_aderidos", "agricultores_com_pagamento_liberado"]:
    df_gs_municipio[col] = pd.to_numeric(df_gs_municipio[col], errors="coerce").fillna(0)
df_gs_municipio["uf"] = df_gs_municipio["uf"].str.upper()
df_gs_municipio = df_gs_municipio[df_gs_municipio["uf"].isin(UF_VALIDAS)].copy()
df_gs_municipio["arquivo_origem"] = arquivo_gs.name

df_gs_uf = (
    df_gs_municipio
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        agricultores_aderidos=("agricultores_aderidos", "sum"),
        agricultores_com_pagamento_liberado=("agricultores_com_pagamento_liberado", "sum"),
        arquivo_origem=("arquivo_origem", "first"),
    )
)

df_gs_municipio.to_csv(POLITICAS_DIR / "garantia_safra_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_gs_uf.to_csv(POLITICAS_DIR / "garantia_safra_uf_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_gs_municipio.shape, df_gs_uf.shape)
display(df_gs_uf.head())

(1215, 10) (11, 6)


,uf,dt_referencia,dt_geracao,agricultores_aderidos,agricultores_com_pagamento_liberado,arquivo_origem
0,AL,2023-2024,2026_04_13,28161,24814,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx
1,AM,2023-2024,2026_04_13,705,274,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx
2,BA,2023-2024,2026_04_13,333616,317522,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx
3,CE,2023-2024,2026_04_13,176528,146296,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx
4,MA,2023-2024,2026_04_13,2530,1362,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx


## 14. PNCF - Limpeza e transformacao

Entrada: aba `DADOS` da base PNCF.

Saidas:
- `pncf_municipio_tratado.csv`
- `pncf_uf_tratado.csv`

In [72]:
arquivo_pncf = arquivos_politicas["pncf"]
if arquivo_pncf is None:
    raise FileNotFoundError("Arquivo PNCF nao encontrado.")

df_pncf_municipio = pd.read_excel(arquivo_pncf, sheet_name="DADOS", engine="openpyxl", na_values=["NA", "na", ""])

colunas_normalizadas = []
for coluna in df_pncf_municipio.columns:
    texto = unicodedata.normalize("NFKD", str(coluna))
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    texto = texto.lower().replace("º", "o").replace("°", "o")
    texto = re.sub(r"[^a-z0-9]+", "_", texto).strip("_")
    colunas_normalizadas.append(texto)
df_pncf_municipio.columns = colunas_normalizadas
df_pncf_municipio = df_pncf_municipio.rename(columns={
    "municipio": "nome_municipio",
    "no_de_operacoes": "numero_operacoes",
    "valor_liberado_r": "valor_liberado",
    "valor_medio_liberado_r_operacao": "valor_medio_liberado",
})

pncf_colunas = ["dt_referencia", "dt_geracao", "cod_ibge", "sigla_uf", "nome_uf", "nome_municipio", "numero_operacoes", "valor_liberado", "valor_medio_liberado"]
faltantes = [col for col in pncf_colunas if col not in df_pncf_municipio.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em PNCF: {faltantes}")
df_pncf_municipio = df_pncf_municipio[pncf_colunas].copy()
for col in ["dt_referencia", "dt_geracao", "cod_ibge", "sigla_uf", "nome_uf", "nome_municipio"]:
    df_pncf_municipio[col] = df_pncf_municipio[col].astype(str).str.strip()
for col in ["numero_operacoes", "valor_liberado", "valor_medio_liberado"]:
    df_pncf_municipio[col] = pd.to_numeric(df_pncf_municipio[col], errors="coerce").fillna(0)
df_pncf_municipio["uf"] = df_pncf_municipio["sigla_uf"].str.upper()
df_pncf_municipio = df_pncf_municipio[df_pncf_municipio["uf"].isin(UF_VALIDAS)].copy()
df_pncf_municipio["arquivo_origem"] = arquivo_pncf.name

df_pncf_uf = (
    df_pncf_municipio
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        numero_operacoes=("numero_operacoes", "sum"),
        valor_liberado=("valor_liberado", "sum"),
        arquivo_origem=("arquivo_origem", "first"),
    )
)
df_pncf_uf["valor_medio_liberado"] = 0.0
mascara_pncf = df_pncf_uf["numero_operacoes"] > 0
df_pncf_uf.loc[mascara_pncf, "valor_medio_liberado"] = df_pncf_uf.loc[mascara_pncf, "valor_liberado"] / df_pncf_uf.loc[mascara_pncf, "numero_operacoes"]

df_pncf_municipio.to_csv(POLITICAS_DIR / "pncf_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_pncf_uf.to_csv(POLITICAS_DIR / "pncf_uf_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_pncf_municipio.shape, df_pncf_uf.shape)
display(df_pncf_uf.head())

(106, 11) (19, 7)


,uf,dt_referencia,dt_geracao,numero_operacoes,valor_liberado,arquivo_origem,valor_medio_liberado
0,AL,2026_04,2026_05_14,10,2532386.00,pncf_2026_04_gerado_em_20260514095754.xlsx,253238.600000
1,BA,2026_04,2026_05_14,43,11359660.42,pncf_2026_04_gerado_em_20260514095754.xlsx,264178.149302
2,CE,2026_04,2026_05_14,65,12501114.24,pncf_2026_04_gerado_em_20260514095754.xlsx,192324.834462
3,ES,2026_04,2026_05_14,4,828048.00,pncf_2026_04_gerado_em_20260514095754.xlsx,207012.000000
4,GO,2026_04,2026_05_14,11,3093500.00,pncf_2026_04_gerado_em_20260514095754.xlsx,281227.272727


## 15. PNRA - Limpeza e transformacao

Entrada: aba `DADOS` da base PNRA.

Saidas:
- `pnra_municipio_tratado.csv`
- `pnra_uf_tratado.csv`

In [73]:
arquivo_pnra = arquivos_politicas["pnra"]
if arquivo_pnra is None:
    raise FileNotFoundError("Arquivo PNRA nao encontrado.")

df_pnra_municipio = pd.read_excel(arquivo_pnra, sheet_name="DADOS", engine="openpyxl", na_values=["NA", "na", ""])
df_pnra_municipio.columns = df_pnra_municipio.columns.astype(str).str.strip()

pnra_colunas = [
    "dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf",
    "numero_familias_pa_criado_diferenciado_pnra", "numero_familias_pa_criado_tradicional_pnra",
    "numero_familias_reconhecimento_pnra", "numero_familias_regularizacao_pnra", "total_numero_familias_pnra",
]
faltantes = [col for col in pnra_colunas if col not in df_pnra_municipio.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes em PNRA: {faltantes}")
df_pnra_municipio = df_pnra_municipio[pnra_colunas].copy()
for col in ["dt_referencia", "dt_geracao", "cod_ibge", "nome_municipio", "uf"]:
    df_pnra_municipio[col] = df_pnra_municipio[col].astype(str).str.strip()
for col in ["numero_familias_pa_criado_diferenciado_pnra", "numero_familias_pa_criado_tradicional_pnra", "numero_familias_reconhecimento_pnra", "numero_familias_regularizacao_pnra", "total_numero_familias_pnra"]:
    df_pnra_municipio[col] = pd.to_numeric(df_pnra_municipio[col], errors="coerce").fillna(0)
df_pnra_municipio["uf"] = df_pnra_municipio["uf"].str.upper()
df_pnra_municipio = df_pnra_municipio[df_pnra_municipio["uf"].isin(UF_VALIDAS)].copy()
df_pnra_municipio["arquivo_origem"] = arquivo_pnra.name

df_pnra_uf = (
    df_pnra_municipio
    .groupby("uf", as_index=False)
    .agg(
        dt_referencia=("dt_referencia", "first"),
        dt_geracao=("dt_geracao", "first"),
        numero_familias_pa_criado_diferenciado_pnra=("numero_familias_pa_criado_diferenciado_pnra", "sum"),
        numero_familias_pa_criado_tradicional_pnra=("numero_familias_pa_criado_tradicional_pnra", "sum"),
        numero_familias_reconhecimento_pnra=("numero_familias_reconhecimento_pnra", "sum"),
        numero_familias_regularizacao_pnra=("numero_familias_regularizacao_pnra", "sum"),
        total_numero_familias_pnra=("total_numero_familias_pnra", "sum"),
        arquivo_origem=("arquivo_origem", "first"),
    )
)

df_pnra_municipio.to_csv(POLITICAS_DIR / "pnra_municipio_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_pnra_uf.to_csv(POLITICAS_DIR / "pnra_uf_tratado.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_pnra_municipio.shape, df_pnra_uf.shape)
display(df_pnra_uf.head())

(5599, 11) (27, 9)


,uf,dt_referencia,dt_geracao,numero_familias_pa_criado_diferenciado_pnra,numero_familias_pa_criado_tradicional_pnra,numero_familias_reconhecimento_pnra,numero_familias_regularizacao_pnra,total_numero_familias_pnra,arquivo_origem
0,AC,2026 até mar,2026_04_15,281.0,0.0,20.0,170.0,471.0,PNRA_2026_2026_04_15.xlsx
1,AL,2026 até mar,2026_04_15,0.0,0.0,5.0,17.0,22.0,PNRA_2026_2026_04_15.xlsx
2,AM,2026 até mar,2026_04_15,498.0,0.0,0.0,130.0,628.0,PNRA_2026_2026_04_15.xlsx
3,AP,2026 até mar,2026_04_15,30.0,0.0,0.0,12.0,42.0,PNRA_2026_2026_04_15.xlsx
4,BA,2026 até mar,2026_04_15,0.0,0.0,0.0,972.0,972.0,PNRA_2026_2026_04_15.xlsx


## 16. Base consolidada de indicadores para relatorio

A base abaixo organiza todos os indicadores em formato longo. Cada linha representa uma politica, um indicador e uma UF. A coluna `valor_brasil` guarda o total nacional do mesmo indicador, `percentual_uf_brasil` permite comparacao direta entre a UF e o Brasil, e `ranking_brasil` registra a posicao da UF entre as UFs no mes de referencia.

Quando uma UF nao possui dado para uma politica ou indicador, a linha e mantida no consolidado com `ranking_brasil = "Sem Informação"`. Isso evita que a ausencia de dado seja confundida com posicao baixa no ranking.

Saida:
- `indicadores_brasil_uf.csv`

In [74]:
registros_indicadores = []

indicadores_caf = [
    ("UFPA", "ufpa", "inteiro"),
    ("CAFs Pessoa Juridica ativos", "cafs_pj_ativo", "inteiro"),
    ("Mulheres em CAF ativo", "mulheres_em_caf_ativo", "inteiro"),
    ("Homens em CAF ativo", "homens_em_caf_ativo", "inteiro"),
]
for indicador, coluna, formato in indicadores_caf:
    valor_brasil = pd.to_numeric(df_caf_uf[coluna], errors="coerce").fillna(0).sum()
    for _, row in df_caf_uf.iterrows():
        valor_uf = float(row[coluna])
        registros_indicadores.append({
            "politica": "CAF", "indicador": indicador, "uf": row["uf"], "valor_uf": valor_uf,
            "valor_brasil": float(valor_brasil), "percentual_uf_brasil": (valor_uf / valor_brasil * 100) if valor_brasil else 0,
            "formato": formato, "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"],
        })

indicadores_pronaf = [
    ("Contratos", "qtd_contratos_total", "inteiro"),
    ("Valor total dos contratos", "valor_total_contratos", "moeda"),
    ("Operacoes", "qtd_operacoes_total", "inteiro"),
    ("Contratos feminino", "qtd_contratos_1_Feminino", "inteiro"),
    ("Valor contratos feminino", "valor_total_contratos_1_Feminino", "moeda"),
    ("Contratos masculino", "qtd_contratos_1_Masculino", "inteiro"),
    ("Valor contratos masculino", "valor_total_contratos_1_Masculino", "moeda"),
]
for indicador, coluna, formato in indicadores_pronaf:
    valor_brasil = pd.to_numeric(df_pronaf_atual_uf[coluna], errors="coerce").fillna(0).sum()
    for _, row in df_pronaf_atual_uf.iterrows():
        valor_uf = float(row[coluna])
        registros_indicadores.append({
            "politica": "PRONAF", "indicador": indicador, "uf": row["uf"], "valor_uf": valor_uf,
            "valor_brasil": float(valor_brasil), "percentual_uf_brasil": (valor_uf / valor_brasil * 100) if valor_brasil else 0,
            "formato": formato, "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"],
        })

indicadores_mais = [("Contratos", "qtd_contratos", "inteiro"), ("Valor total dos contratos", "valor_total_contratos", "moeda")]
for indicador, coluna, formato in indicadores_mais:
    valor_brasil = pd.to_numeric(df_mais_uf[coluna], errors="coerce").fillna(0).sum()
    for _, row in df_mais_uf.iterrows():
        valor_uf = float(row[coluna])
        registros_indicadores.append({
            "politica": "Mais Alimentos", "indicador": indicador, "uf": row["uf"], "valor_uf": valor_uf,
            "valor_brasil": float(valor_brasil), "percentual_uf_brasil": (valor_uf / valor_brasil * 100) if valor_brasil else 0,
            "formato": formato, "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"],
        })

indicadores_ater = [("Familias com ATER iniciada no ano", "familias_com_ater_iniciada_no_ano", "inteiro"), ("Familias com ATER recebida no ano", "familias_com_ater_recebida_no_ano", "inteiro")]
for indicador, coluna, formato in indicadores_ater:
    valor_brasil = pd.to_numeric(df_ater_atual_uf[coluna], errors="coerce").fillna(0).sum()
    for _, row in df_ater_atual_uf.iterrows():
        valor_uf = float(row[coluna])
        registros_indicadores.append({
            "politica": "ATER", "indicador": indicador, "uf": row["uf"], "valor_uf": valor_uf,
            "valor_brasil": float(valor_brasil), "percentual_uf_brasil": (valor_uf / valor_brasil * 100) if valor_brasil else 0,
            "formato": formato, "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"],
        })

indicadores_gs = [("Agricultores aderidos", "agricultores_aderidos", "inteiro"), ("Agricultores com pagamento liberado", "agricultores_com_pagamento_liberado", "inteiro")]
for indicador, coluna, formato in indicadores_gs:
    valor_brasil = pd.to_numeric(df_gs_uf[coluna], errors="coerce").fillna(0).sum()
    for _, row in df_gs_uf.iterrows():
        valor_uf = float(row[coluna])
        registros_indicadores.append({
            "politica": "Garantia-Safra", "indicador": indicador, "uf": row["uf"], "valor_uf": valor_uf,
            "valor_brasil": float(valor_brasil), "percentual_uf_brasil": (valor_uf / valor_brasil * 100) if valor_brasil else 0,
            "formato": formato, "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"],
        })

indicadores_pncf = [("Operacoes", "numero_operacoes", "inteiro"), ("Valor liberado", "valor_liberado", "moeda"), ("Valor medio liberado por operacao", "valor_medio_liberado", "moeda")]
for indicador, coluna, formato in indicadores_pncf:
    if coluna == "valor_medio_liberado":
        total_operacoes = pd.to_numeric(df_pncf_uf["numero_operacoes"], errors="coerce").fillna(0).sum()
        total_valor = pd.to_numeric(df_pncf_uf["valor_liberado"], errors="coerce").fillna(0).sum()
        valor_brasil = total_valor / total_operacoes if total_operacoes else 0
    else:
        valor_brasil = pd.to_numeric(df_pncf_uf[coluna], errors="coerce").fillna(0).sum()
    for _, row in df_pncf_uf.iterrows():
        valor_uf = float(row[coluna])
        registros_indicadores.append({
            "politica": "PNCF", "indicador": indicador, "uf": row["uf"], "valor_uf": valor_uf,
            "valor_brasil": float(valor_brasil), "percentual_uf_brasil": (valor_uf / valor_brasil * 100) if valor_brasil else 0,
            "formato": formato, "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"],
        })

indicadores_pnra = [
    ("Familias em PA criado diferenciado", "numero_familias_pa_criado_diferenciado_pnra", "inteiro"),
    ("Familias em PA criado tradicional", "numero_familias_pa_criado_tradicional_pnra", "inteiro"),
    ("Familias em reconhecimento", "numero_familias_reconhecimento_pnra", "inteiro"),
    ("Familias em regularizacao", "numero_familias_regularizacao_pnra", "inteiro"),
    ("Total de familias", "total_numero_familias_pnra", "inteiro"),
]
for indicador, coluna, formato in indicadores_pnra:
    valor_brasil = pd.to_numeric(df_pnra_uf[coluna], errors="coerce").fillna(0).sum()
    for _, row in df_pnra_uf.iterrows():
        valor_uf = float(row[coluna])
        registros_indicadores.append({
            "politica": "PNRA", "indicador": indicador, "uf": row["uf"], "valor_uf": valor_uf,
            "valor_brasil": float(valor_brasil), "percentual_uf_brasil": (valor_uf / valor_brasil * 100) if valor_brasil else 0,
            "formato": formato, "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"],
        })

df_indicadores_brasil_uf = pd.DataFrame(registros_indicadores)

# Garante que todos os indicadores tenham uma linha para cada UF.
# Algumas politicas atuais nao trazem registros para todas as UFs. Nesses casos, a UF deve aparecer no relatorio como "Sem Informação" no ranking, e nao desaparecer da tabela.
df_indicadores_brasil_uf["uf"] = df_indicadores_brasil_uf["uf"].astype(str).str.strip().str.upper()
df_indicadores_brasil_uf["valor_uf"] = pd.to_numeric(df_indicadores_brasil_uf["valor_uf"], errors="coerce")
df_indicadores_brasil_uf["valor_brasil"] = pd.to_numeric(df_indicadores_brasil_uf["valor_brasil"], errors="coerce").fillna(0)
df_indicadores_brasil_uf["percentual_uf_brasil"] = pd.to_numeric(df_indicadores_brasil_uf["percentual_uf_brasil"], errors="coerce")

colunas_indicador = ["politica", "indicador", "formato", "dt_referencia", "dt_geracao", "arquivo_origem", "valor_brasil"]
df_chaves_indicadores = df_indicadores_brasil_uf[colunas_indicador].drop_duplicates().copy()
df_ufs_grade = df_ufs[["sigla"]].drop_duplicates().rename(columns={"sigla": "uf"}).copy()

df_chaves_indicadores["_chave_grade"] = 1
df_ufs_grade["_chave_grade"] = 1
df_grade_indicadores = (
    df_chaves_indicadores
    .merge(df_ufs_grade, on="_chave_grade", how="inner")
    .drop(columns=["_chave_grade"])
)

df_indicadores_brasil_uf = df_grade_indicadores.merge(
    df_indicadores_brasil_uf,
    on=[*colunas_indicador, "uf"],
    how="left",
)
df_indicadores_brasil_uf["tem_dado_uf"] = df_indicadores_brasil_uf["valor_uf"].notna()
df_indicadores_brasil_uf["percentual_uf_brasil"] = (
    df_indicadores_brasil_uf["valor_uf"] / df_indicadores_brasil_uf["valor_brasil"] * 100
)
df_indicadores_brasil_uf.loc[df_indicadores_brasil_uf["valor_brasil"] == 0, "percentual_uf_brasil"] = pd.NA

# Ranking Brasil no mes de referencia.
# O calculo considera apenas UFs com dado disponivel. UFs sem dado recebem "Sem Informação".
# Empates recebem a mesma posicao pelo metodo "min", mantendo a leitura usual de ranking institucional.
df_indicadores_brasil_uf["_dt_referencia_ranking"] = df_indicadores_brasil_uf["dt_referencia"].fillna("sem_referencia").astype(str)
df_indicadores_brasil_uf["_ranking_calculado"] = pd.NA
mascara_com_dado = df_indicadores_brasil_uf["tem_dado_uf"] & df_indicadores_brasil_uf["valor_uf"].notna()
df_indicadores_brasil_uf.loc[mascara_com_dado, "_ranking_calculado"] = (
    df_indicadores_brasil_uf.loc[mascara_com_dado]
    .groupby(["politica", "indicador", "_dt_referencia_ranking"])["valor_uf"]
    .rank(method="min", ascending=False)
)
df_indicadores_brasil_uf["ranking_brasil"] = "Sem Informação"
df_indicadores_brasil_uf.loc[mascara_com_dado, "ranking_brasil"] = (
    df_indicadores_brasil_uf.loc[mascara_com_dado, "_ranking_calculado"]
    .astype("Int64")
    .astype(str)
)
df_indicadores_brasil_uf = df_indicadores_brasil_uf.drop(columns=["_dt_referencia_ranking", "_ranking_calculado"])

df_indicadores_brasil_uf = df_indicadores_brasil_uf.merge(df_ufs, left_on="uf", right_on="sigla", how="left")
df_indicadores_brasil_uf = df_indicadores_brasil_uf[["politica", "indicador", "uf", "codigo_ibge", "nome_uf", "valor_uf", "valor_brasil", "percentual_uf_brasil", "ranking_brasil", "formato", "dt_referencia", "dt_geracao", "arquivo_origem"]]

df_indicadores_brasil_uf.to_csv(CONSOLIDADO_DIR / "indicadores_brasil_uf.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_indicadores_brasil_uf.shape)
display(df_indicadores_brasil_uf.head(20))

(675, 13)


,politica,indicador,uf,codigo_ibge,nome_uf,valor_uf,valor_brasil,percentual_uf_brasil,ranking_brasil,formato,dt_referencia,dt_geracao,arquivo_origem
0,CAF,UFPA,AC,12,Acre,33690.0,3929314.0,0.857402,22,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
1,CAF,UFPA,AL,27,Alagoas,91652.0,3929314.0,2.332519,12,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
2,CAF,UFPA,AP,16,Amapá,21820.0,3929314.0,0.555313,24,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
3,CAF,UFPA,AM,13,Amazonas,66861.0,3929314.0,1.701595,14,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
4,CAF,UFPA,BA,29,Bahia,725958.0,3929314.0,18.475439,1,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
5,CAF,UFPA,CE,23,Ceará,420317.0,3929314.0,10.696956,2,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
6,CAF,UFPA,DF,53,Distrito Federal,4010.0,3929314.0,0.102053,27,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
7,CAF,UFPA,ES,32,Espírito Santo,55465.0,3929314.0,1.411570,17,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
8,CAF,UFPA,GO,52,Goiás,41459.0,3929314.0,1.055121,19,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...
9,CAF,UFPA,MA,21,Maranhão,359780.0,3929314.0,9.156306,3,inteiro,2026_04,2026_05_12,caf_dap_ativos_ate_2026_04_gerado_em_202605121...


## 17. Base consolidada de series historicas

Esta base junta as series anuais de PRONAF e ATER em formato longo. Ela sera usada posteriormente para graficos historicos no relatorio da alta gestao.

Saida:
- `series_historicas_pronaf_ater_uf.csv`

In [75]:
registros_series = []

if not df_pronaf_historico_anual_uf.empty:
    for _, row in df_pronaf_historico_anual_uf.iterrows():
        registros_series.append({"politica": "PRONAF", "ano": row["ano"], "uf": row["uf"], "indicador": "Contratos", "valor": row["qtd_contratos_total"], "formato": "inteiro", "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"]})
        registros_series.append({"politica": "PRONAF", "ano": row["ano"], "uf": row["uf"], "indicador": "Valor total dos contratos", "valor": row["valor_total_contratos"], "formato": "moeda", "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"]})
        registros_series.append({"politica": "PRONAF", "ano": row["ano"], "uf": row["uf"], "indicador": "Operacoes", "valor": row["qtd_operacoes_total"], "formato": "inteiro", "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"]})

if not df_ater_historico_anual_uf.empty:
    for _, row in df_ater_historico_anual_uf.iterrows():
        registros_series.append({"politica": "ATER", "ano": row["ano"], "uf": row["uf"], "indicador": "Familias com ATER iniciada no ano", "valor": row["familias_com_ater_iniciada_no_ano"], "formato": "inteiro", "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"]})
        registros_series.append({"politica": "ATER", "ano": row["ano"], "uf": row["uf"], "indicador": "Familias com ATER recebida no ano", "valor": row["familias_com_ater_recebida_no_ano"], "formato": "inteiro", "dt_referencia": row["dt_referencia"], "dt_geracao": row["dt_geracao"], "arquivo_origem": row["arquivo_origem"]})

df_series_historicas = pd.DataFrame(registros_series)
if not df_series_historicas.empty:
    df_series_historicas = df_series_historicas.merge(df_ufs, left_on="uf", right_on="sigla", how="left")
    df_series_historicas = df_series_historicas[["politica", "ano", "uf", "codigo_ibge", "nome_uf", "indicador", "valor", "formato", "dt_referencia", "dt_geracao", "arquivo_origem"]]
else:
    df_series_historicas = pd.DataFrame(columns=["politica", "ano", "uf", "codigo_ibge", "nome_uf", "indicador", "valor", "formato", "dt_referencia", "dt_geracao", "arquivo_origem"])

df_series_historicas.to_csv(CONSOLIDADO_DIR / "series_historicas_pronaf_ater_uf.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(df_series_historicas.shape)
display(df_series_historicas.head(20))

(1307, 11)


,politica,ano,uf,codigo_ibge,nome_uf,indicador,valor,formato,dt_referencia,dt_geracao,arquivo_origem
0,PRONAF,2015,AC,12,Acre,Contratos,5.501000e+03,inteiro,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
1,PRONAF,2015,AC,12,Acre,Valor total dos contratos,1.110044e+08,moeda,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
2,PRONAF,2015,AC,12,Acre,Operacoes,5.501000e+03,inteiro,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
3,PRONAF,2015,AL,27,Alagoas,Contratos,5.034000e+04,inteiro,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
4,PRONAF,2015,AL,27,Alagoas,Valor total dos contratos,1.977084e+08,moeda,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
5,PRONAF,2015,AL,27,Alagoas,Operacoes,5.034000e+04,inteiro,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
6,PRONAF,2015,AM,13,Amazonas,Contratos,3.382000e+03,inteiro,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
7,PRONAF,2015,AM,13,Amazonas,Valor total dos contratos,5.350340e+07,moeda,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
8,PRONAF,2015,AM,13,Amazonas,Operacoes,3.382000e+03,inteiro,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx
9,PRONAF,2015,AP,16,Amapá,Contratos,8.040000e+02,inteiro,2015_12,2026_04_29,pronaf_gaia_historico_anual_2015.xlsx


## 18. Resumo final da ETL

A ultima etapa lista os arquivos CSV gerados nesta execucao. Esses arquivos sao a interface entre a ETL e o notebook futuro de relatorio.

In [76]:
arquivos_csv_gerados = sorted(OUTPUT_ROOT_DIR.rglob("*.csv"))
df_saida = pd.DataFrame([
    {
        "arquivo": arquivo.name,
        "subpasta": str(arquivo.parent.relative_to(OUTPUT_ROOT_DIR)),
        "caminho": str(arquivo),
        "tamanho_kb": round(arquivo.stat().st_size / 1024, 2),
    }
    for arquivo in arquivos_csv_gerados
])
df_saida.to_csv(LOGS_DIR / "resumo_csv_gerados.csv", sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

print(f"ETL concluida. CSVs gerados em: {OUTPUT_ROOT_DIR}")
display(df_saida)

ETL concluida. CSVs gerados em: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_intermediarios\202605


,arquivo,subpasta,caminho,tamanho_kb
0,indicadores_brasil_uf.csv,consolidado,C:\Users\marce\OneDrive - Ministério da Agricu...,99.18
1,series_historicas_pronaf_ater_uf.csv,consolidado,C:\Users\marce\OneDrive - Ministério da Agricu...,165.97
2,ater_historico_anual_uf.csv,historico,C:\Users\marce\OneDrive - Ministério da Agricu...,19.22
3,ater_historico_municipio_tratado.csv,historico,C:\Users\marce\OneDrive - Ministério da Agricu...,965.04
4,pronaf_historico_anual_uf.csv,historico,C:\Users\marce\OneDrive - Ministério da Agricu...,26.56
5,pronaf_historico_municipio_tratado.csv,historico,C:\Users\marce\OneDrive - Ministério da Agricu...,11295.42
6,estrutura_abas_colunas.csv,logs,C:\Users\marce\OneDrive - Ministério da Agricu...,21.50
7,inventario_arquivos.csv,logs,C:\Users\marce\OneDrive - Ministério da Agricu...,7.19
8,resumo_csv_gerados.csv,logs,C:\Users\marce\OneDrive - Ministério da Agricu...,5.56
9,ufs_referencia.csv,logs,C:\Users\marce\OneDrive - Ministério da Agricu...,0.50
